In [1]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
print("Libraries loaded ✓")

Libraries loaded ✓


In [2]:
orders    = pd.read_csv('data/olist_orders_dataset.csv')
items     = pd.read_csv('data/olist_order_items_dataset.csv')
customers = pd.read_csv('data/olist_customers_dataset.csv')
products  = pd.read_csv('data/olist_products_dataset.csv')
sellers   = pd.read_csv('data/olist_sellers_dataset.csv')
reviews   = pd.read_csv('data/olist_order_reviews_dataset.csv')
payments  = pd.read_csv('data/olist_order_payments_dataset.csv')
cat_trans = pd.read_csv('data/product_category_name_translation.csv')

for name, df in [('orders', orders), ('items', items), ('customers', customers),
                 ('products', products), ('sellers', sellers),
                 ('reviews', reviews), ('payments', payments)]:
    print(f"{name:12} → {df.shape[0]:>7,} rows  |  {df.shape[1]} columns")

orders       →  99,441 rows  |  8 columns
items        → 112,650 rows  |  7 columns
customers    →  99,441 rows  |  5 columns
products     →  32,951 rows  |  9 columns
sellers      →   3,095 rows  |  4 columns
reviews      →  99,224 rows  |  7 columns
payments     → 103,886 rows  |  5 columns


In [3]:
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    orders[col] = pd.to_datetime(orders[col])

items['shipping_limit_date'] = pd.to_datetime(items['shipping_limit_date'])

print("Datetime columns fixed ✓")
print("\nNull counts in order dates:")
print(orders[date_cols].isnull().sum())

Datetime columns fixed ✓

Null counts in order dates:
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64


In [4]:
# Reviews: missing comment text is fine, just fill with empty string
reviews['review_comment_message'] = reviews['review_comment_message'].fillna('')
reviews['review_comment_title']   = reviews['review_comment_title'].fillna('')

# Products: missing category → label as 'unknown'
products['product_category_name'] = products['product_category_name'].fillna('unknown')

# Products: missing dimensions → fill with median
dim_cols = ['product_weight_g', 'product_length_cm',
            'product_height_cm', 'product_width_cm']
for col in dim_cols:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)
    print(f"  {col}: filled nulls with median → {median_val:.1f}")

print("\nNull handling done ✓")

  product_weight_g: filled nulls with median → 700.0
  product_length_cm: filled nulls with median → 25.0
  product_height_cm: filled nulls with median → 13.0
  product_width_cm: filled nulls with median → 20.0

Null handling done ✓


In [5]:
before = len(reviews)
reviews = reviews.drop_duplicates(subset='review_id')
print(f"Reviews: removed {before - len(reviews)} duplicate rows")

before = len(orders)
orders = orders.drop_duplicates(subset='order_id')
print(f"Orders:  removed {before - len(orders)} duplicate rows")

Reviews: removed 814 duplicate rows
Orders:  removed 0 duplicate rows


In [6]:
# Flag delivered orders
orders['was_delivered'] = (orders['order_status'] == 'delivered')

# Delivery time in days (only meaningful for delivered orders)
orders['delivery_days'] = (
    orders['order_delivered_customer_date'] -
    orders['order_purchase_timestamp']
).dt.days

# Was it late? (actual delivery vs what was promised)
orders['was_late'] = (
    orders['order_delivered_customer_date'] >
    orders['order_estimated_delivery_date']
)

# Quick stats
delivered = orders[orders['was_delivered']]
print(f"Delivered orders  : {orders['was_delivered'].sum():,}")
print(f"Average delivery  : {delivered['delivery_days'].mean():.1f} days")
print(f"Late deliveries   : {orders['was_late'].sum():,} ({orders['was_late'].mean()*100:.1f}%)")
print(f"Fastest delivery  : {delivered['delivery_days'].min():.0f} days")
print(f"Slowest delivery  : {delivered['delivery_days'].max():.0f} days")

Delivered orders  : 96,478
Average delivery  : 12.1 days
Late deliveries   : 7,827 (7.9%)
Fastest delivery  : 0 days
Slowest delivery  : 209 days


In [7]:
products = products.merge(cat_trans, on='product_category_name', how='left')
products['category_en'] = (
    products['product_category_name_english']
    .fillna(products['product_category_name'])
)

print(f"Categories translated ✓")
print(f"Total unique categories: {products['category_en'].nunique()}")

Categories translated ✓
Total unique categories: 74


In [8]:
# Aggregate payments per order (some orders have multiple payment methods)
payments_agg = payments.groupby('order_id').agg(
    payment_value  = ('payment_value', 'sum'),
    payment_type   = ('payment_type',  'first'),
    installments   = ('payment_installments', 'max')
).reset_index()

# Aggregate reviews per order (take most recent if duplicates)
reviews_agg = reviews.sort_values('review_answer_timestamp').drop_duplicates(
    subset='order_id', keep='last'
)[['order_id', 'review_score']]

# Master merge
master = (
    orders
    .merge(items,        on='order_id',    how='left')
    .merge(customers,    on='customer_id', how='left')
    .merge(products,     on='product_id',  how='left')
    .merge(sellers,      on='seller_id',   how='left')
    .merge(payments_agg, on='order_id',    how='left')
    .merge(reviews_agg,  on='order_id',    how='left')
)

print(f"Master table shape  : {master.shape}")
print(f"Unique orders       : {master['order_id'].nunique():,}")
print(f"Unique customers    : {master['customer_id'].nunique():,}")
print(f"Unique sellers      : {master['seller_id'].nunique():,}")

Master table shape  : (113425, 38)
Unique orders       : 99,441
Unique customers    : 99,441
Unique sellers      : 3,095


In [9]:
print("=" * 45)
print("        CLEANING SUMMARY")
print("=" * 45)
print(f"Total rows          : {len(master):,}")
print(f"Unique orders       : {master['order_id'].nunique():,}")
print(f"Unique customers    : {master['customer_id'].nunique():,}")
print(f"Unique sellers      : {master['seller_id'].nunique():,}")
print(f"Unique categories   : {master['category_en'].nunique():,}")
print(f"Date range          : {master['order_purchase_timestamp'].min().date()} "
      f"→ {master['order_purchase_timestamp'].max().date()}")
print(f"Avg review score    : {master['review_score'].mean():.2f} / 5")
print(f"Avg payment value   : R$ {master['payment_value'].mean():.2f}")
print(f"Avg delivery days   : {master['delivery_days'].mean():.1f}")
print(f"Late delivery rate  : {master['was_late'].mean()*100:.1f}%")
print("=" * 45)

master.to_csv('data/olist_master.csv', index=False)
print("\nSaved → data/olist_master.csv ✓")

        CLEANING SUMMARY
Total rows          : 113,425
Unique orders       : 99,441
Unique customers    : 99,441
Unique sellers      : 3,095
Unique categories   : 74
Date range          : 2016-09-04 → 2018-10-17
Avg review score    : 4.02 / 5
Avg payment value   : R$ 180.48
Avg delivery days   : 12.0
Late delivery rate  : 7.7%

Saved → data/olist_master.csv ✓
